In [1]:
import os
iskaggle = os.environ.get('KAGGLE_KERNEL_RUN_TYPE', '')
RANDOMSEED=1727

In [ ]:
import torch,random
import numpy as np
def random_seed(seed_value, use_cuda=False):
    np.random.seed(seed_value) # cpu vars
    torch.manual_seed(seed_value) # cpu vars
    random.seed(seed_value) # Python
    if use_cuda: 
        torch.cuda.manual_seed(seed_value)
        torch.cuda.manual_seed_all(seed_value) # gpu vars
        torch.backends.cudnn.deterministic = True  #needed
        torch.backends.cudnn.benchmark = False
random_seed(1727)

In [3]:
from pathlib import Path

cred_path = Path('~/.kaggle/access_token').expanduser()
if not cred_path.exists():
    cred_path.parent.mkdir(exist_ok=True)
    cred_path.write_text("KGAT_9f6b15aaf6f7637b8497dfb3c56c079e")
    cred_path.chmod(0o600)

In [4]:
path = Path('ml-olympiad-machine-translation-french-wolof')

In [ ]:
if not iskaggle:
    if not path.exists():
        import zipfile,kaggle
        kaggle.api.competition_download_cli(str(path))
        zipfile.ZipFile(f'{path}.zip').extractall(path)
else:
    # /kaggle/input/competitions/ml-olympiad-machine-translation-french-wolof/train.csv
    path = Path('/kaggle/input/competitions/ml-olympiad-machine-translation-french-wolof')

# %pip install -q datasets
!dir /o:g /w {path}
# !ls {path}

In [36]:
import pandas as pd
train=pd.read_csv(path/"train.csv")
test=pd.read_csv(path/"test.csv")

# drop the `id` col for training data ONLY, we need the id for test preds later
train.drop(columns=["ID"],inplace=True)

In [37]:
# Your code here to load data
# Expected format:
# train_fr = ["je mange du riz", "il fait beau", ...]  
# train_wo = ["maam na ceeb", "beautiful na", ...]
# test_fr = ["je vais au marché", ...]

train_fr=train["FRENCH"].astype(str).to_list()
train_wo=train["WOLOF"].astype(str).to_list()
test_fr=test["FRENCH"].astype(str).to_list()

train_fr[:5]
# train_wo[:5]

fr_tokens = [x.strip().split() for x in train_fr]
wo_tokens = [x.strip().split() for x in train_wo]

# build FastText embeddings

In [38]:
from gensim.models import FastText

fr_model = FastText(
    vector_size=256,
    window=5,
    min_count=2,
    epochs=50,
    sg=1,
    sentences=fr_tokens
)
wo_model = FastText(
    vector_size=256,
    window=5,
    min_count=2,
    epochs=50,
    sg=1,
    sentences=wo_tokens
)

In [39]:
def sentence_embedding(sentence, model):
    """Average word embeddings to get sentence embedding"""
    words = sentence.split()
    word_vectors = []
    for word in words:
        if word in model.wv:
            word_vectors.append(model.wv[word])
    if len(word_vectors) == 0:
        return np.zeros(word_vectors)
    return np.mean(word_vectors, axis=0)

In [40]:
fr_train_embeddings = np.array([
    sentence_embedding(sent, fr_model) 
    for sent in train_fr
])
wo_train_embeddings = np.array([
    sentence_embedding(sent, wo_model) 
    for sent in train_wo
])

In [41]:
fr_train_embeddings = fr_train_embeddings / (np.linalg.norm(fr_train_embeddings, axis=1, keepdims=True) + 1e-8)
wo_train_embeddings = wo_train_embeddings / (np.linalg.norm(wo_train_embeddings, axis=1, keepdims=True) + 1e-8)

In [ ]:
from sklearn.neighbors import NearestNeighbors

knn_fr = NearestNeighbors(n_neighbors=5, metric='cosine')
knn_fr.fit(fr_train_embeddings)

In [ ]:
print("Translating test sentences...")
preds = []

for test_sent in test_fr:
    # Get test sentence embedding
    test_emb = sentence_embedding(test_sent, fr_model)
    test_emb = test_emb / (np.linalg.norm(test_emb) + 1e-8)
    test_emb = test_emb.reshape(1, -1)
    
    # Find similar French sentences in training data
    distances, indices = knn_fr.kneighbors(test_emb)
    
    # Get corresponding Wolof translations
    retrieved_wo_sentences = [train_wo[idx] for idx in indices[0]]
    
    # Simple approach: just use the closest match
    preds.append(retrieved_wo_sentences[0])

In [44]:
subs_df=pd.DataFrame({
  "ID":test["ID"],
  "Target":preds,
})

In [ ]:
# subs_df.head()
subs_df.shape

In [46]:
subs_df.to_csv("submission.csv",index=False)

In [ ]:
subs_df.head()